In [1]:
# Code Block 1
# Referenced/Used material from AI healthcare class
# Referenced/Used previous work completed by me for AI healthcare class
from google.cloud import bigquery
from google.colab import auth, userdata
from sklearn.model_selection import train_test_split
from pathlib import Path

auth.authenticate_user()

# Referenced/Used: https://stackoverflow.com/questions/66631333/how-do-i-set-environment-variables-in-google-colab
project_id = userdata.get("PROJECT_ID")
# End of: Referenced/Used: https://stackoverflow.com/questions/66631333/how-do-i-set-environment-variables-in-google-colab

client = bigquery.Client(project=project_id)
# End of: Referenced/Used material from AI healthcare class
# End of: Referenced/Used previous work completed by me for AI healthcare class

In [2]:
# Code Block 2
# Referenced/Used material from AI healthcare class
# Referenced/Used previous work completed by me for AI healthcare class
# Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (SQL CASE query help)
# Referenced/Used: https://www.programiz.com/sql/like-operator
radiology_query = """
SELECT
radiology.subject_id AS subject_id,
radiology.hadm_id AS hadm_id,
radiology.text AS notes,
d_icd_diagnoses.icd_version AS icd_version,
d_icd_diagnoses.icd_code AS icd_code,
d_icd_diagnoses.long_title AS title,
CASE
    WHEN LOWER(d_icd_diagnoses.long_title) LIKE '%malignant%' THEN 1
    ELSE 0
END AS is_malignant_diagnoses
FROM `physionet-data.mimiciv_note.radiology` AS radiology
JOIN `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` AS diagnoses_icd
ON radiology.subject_id = diagnoses_icd.subject_id
AND radiology.hadm_id = diagnoses_icd.hadm_id
JOIN `physionet-data.mimiciv_3_1_hosp.d_icd_diagnoses` AS d_icd_diagnoses
ON d_icd_diagnoses.icd_code = diagnoses_icd.icd_code
AND d_icd_diagnoses.icd_version = diagnoses_icd.icd_version
WHERE (d_icd_diagnoses.icd_version = 9 AND diagnoses_icd.icd_version = 9
AND LOWER(d_icd_diagnoses.long_title) LIKE '%tumor%')
AND (LOWER(d_icd_diagnoses.long_title) LIKE '%malignant%'
OR LOWER(d_icd_diagnoses.long_title) LIKE '%benign%');
"""
# End of: Referenced/Used: https://www.programiz.com/sql/like-operator
# End of: Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (SQL CASE query help)

# Referenced/Used: https://docs.cloud.google.com/python/docs/reference/bigquery/latest
result = client.query_and_wait(radiology_query)
df = result.to_dataframe()
# End of: Referenced/Used: https://docs.cloud.google.com/python/docs/reference/bigquery/latest
# End of: Referenced/Used material from AI healthcare class
# End of: Referenced/Used previous work completed by me for AI healthcare class

In [3]:
# Code Block 3
# Referenced/Used material from AI healthcare class
# Referenced/Used previous work completed by me for AI healthcare class
# Referenced/Used: https://codezup.com/natural-language-processing-sentiment-analysis-scikit-learn/
# Referenced/Used: https://stackabuse.com/python-for-nlp-sentiment-analysis-with-scikit-learn/
# Referenced/Used: https://www.geeksforgeeks.org/python/python-pandas-dataframe-dropna/
# Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
df = df.dropna(axis=0, subset=['notes', 'is_malignant_diagnoses'])
df = df[['notes', 'is_malignant_diagnoses']]
x_train, y_train, x_test, y_test = train_test_split(df['notes'], df['is_malignant_diagnoses'], train_size = 0.7, random_state=42)
# End of: Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

train_path = Path("/content/data/train")
validation_path = Path("/content/data/validation")
train_path.mkdir(exist_ok=True, parents=True)
validation_path.mkdir(exist_ok=True, parents=True)


x_train.to_csv(train_path / "features.csv")
x_test.to_csv(train_path / "results.csv")

y_train.to_csv(validation_path / "features.csv")
y_test.to_csv(validation_path / "results.csv")
# End of: Referenced/Used: https://www.geeksforgeeks.org/python/python-pandas-dataframe-dropna/
# End of: Referenced/Used: https://codezup.com/natural-language-processing-sentiment-analysis-scikit-learn/
# End of: Referenced/Used: https://stackabuse.com/python-for-nlp-sentiment-analysis-with-scikit-learn/
# End of: Referenced/Used previous work completed by me for AI healthcare class
# End of: Referenced/Used material from AI healthcare class
# End of: Referenced/Used previous work completed by me for AI healthcare class

In [4]:
# Code Block 4
# Referenced/Used material from AI healthcare class
# Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (General help, Help with parse_args)
# Referenced/Used previous work completed by me for AI healthcare class
# Referenced/Used: https://stackabuse.com/python-for-nlp-sentiment-analysis-with-scikit-learn/
# Referenced/Used: https://codezup.com/natural-language-processing-sentiment-analysis-scikit-learn/
# Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesClassifier.html
# Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html
# Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html
# Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
# Referenced/Used: https://www.geeksforgeeks.org/machine-learning/confusion-matrix-machine-learning/
# Referenced/Used: https://www.geeksforgeeks.org/machine-learning/f1-score-in-machine-learning/
from pathlib import Path
import pandas as pd
import argparse
from enum import Enum
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, BaggingClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Getting scores from sklearn for training and validation)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Enum usage)
class Classifier(Enum):
    ada_boost = 1
    random_forest = 2
    bagging = 3

# Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Setting up and crafting parse_args)
def parse_args():
    parser = argparse.ArgumentParser(description="Train a malignant/benign tumor diagnoses classifier using analysis on radiology notes.")
    parser.add_argument(
        "--model",
        choices=[c.name for c in Classifier],
        default=Classifier.ada_boost,
        type=str,
        help="Classifier to use",
    )
    return parser.parse_args()

def classifier_scores(truth, prediction):

    accuracy_val = accuracy_score(truth, prediction)
    precision_val = precision_score(truth, prediction)
    recall_val = recall_score(truth, prediction)
    f1_val = f1_score(truth, prediction)

    return (precision_val, recall_val, accuracy_val, f1_val)

# Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Help with plotting confusion matrix)
# Referenced/Used: https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.savefig.html
# Referenced/Used: https://pythonguides.com/scikit-learn-confusion-matrix/
# Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html#sklearn.metrics.confusion_matrix
# Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay
def graph_confusion_matrix(truth, prediction, classifier_name="test"):
    results = confusion_matrix(truth, prediction)
    display = ConfusionMatrixDisplay(results, display_labels=["benign", "malignant"])
    display.plot()
    plt.title(f"{classifier_name} classification performance")
    plt.savefig(f"./{classifier_name}.png")
# End of: Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html#sklearn.metrics.confusion_matrix
# End of: Referenced/Used: https://pythonguides.com/scikit-learn-confusion-matrix/
# End of: Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay
# End of: Referenced/Used: https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.savefig.html
# End of: Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Help with plotting confusion matrix)

# Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html
# Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
# Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html#sklearn.ensemble.AdaBoostClassifier
# Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Help with training and predicting script)
def tutorial(classifier_id):
    train_features_path = Path("./data/train/features.csv")
    train_results_path = Path("./data/train/results.csv")
# Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Read csv to dataframe)
    train_features_df = pd.read_csv(train_features_path)
    train_results_df = pd.read_csv(train_results_path)
# End of: Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Read csv to dataframe)

# Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Understaning and usage of TfidfVectorizer)
    vectorizer = TfidfVectorizer(stop_words='english', max_features=2000)
    transformed_train_features_df = vectorizer.fit_transform(train_features_df["notes"])
# End of: Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Understaning and usage of TfidfVectorizer)

    if Classifier.ada_boost.name == classifier_id:
        classifier = AdaBoostClassifier(random_state=42)
    elif Classifier.random_forest.name == classifier_id:
        classifier = RandomForestClassifier(random_state=42)
    else:
        classifier = BaggingClassifier(random_state=42)
# End of: Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Setting up and crafting parse_args)
# End of: Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Enum usage)
    classifier = classifier.fit(transformed_train_features_df, train_results_df["is_malignant_diagnoses"])

    train_prediction = classifier.predict(transformed_train_features_df)

    validation_features_path = Path("./data/validation/features.csv")
    validation_results_path = Path("./data/validation/results.csv")

# Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Read csv to dataframe)
    validation_features_df = pd.read_csv(validation_features_path)
    validation_results_df = pd.read_csv(validation_results_path)
# End of: Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Read csv to dataframe)

    transformed_validation_features_df = vectorizer.transform(validation_features_df["notes"])

    validation_prediction = classifier.predict(transformed_validation_features_df)

    train_scores = classifier_scores(train_results_df["is_malignant_diagnoses"], train_prediction)
    validation_scores = classifier_scores(validation_results_df["is_malignant_diagnoses"], validation_prediction)

    classifier_name = classifier.__class__.__name__
# Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (truncating decimals)
    train_output = f"""Training Scores - {classifier_name}:

Precision: {train_scores[0]:.5f}, Recall: {train_scores[1]:.5f}, Accuracy: {train_scores[2]:.5f}, F1: {train_scores[3]:.5f}
    """

    validation_output = f"""Validation Scores - {classifier_name}:

Precision: {validation_scores[0]:.5f}, Recall: {validation_scores[1]:.5f}, Accuracy: {validation_scores[2]:.5f}, F1: {validation_scores[3]:.5f}
    """
# End of: Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (truncating decimals)
# End of: Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Getting scores from sklearn for training and validation)

    print("===================================================================")
    print(train_output)
    print("===================================================================")
    print(validation_output)
    print("===================================================================")

    graph_confusion_matrix(validation_results_df["is_malignant_diagnoses"], validation_prediction, classifier_name)
# End of: Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (Help with training and predicting script)
# End of: Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
# End of: Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html#sklearn.ensemble.AdaBoostClassifier
# End of: Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html

# End of: Referenced/Used: https://codezup.com/natural-language-processing-sentiment-analysis-scikit-learn/
# End of: Referenced/Used: https://stackabuse.com/python-for-nlp-sentiment-analysis-with-scikit-learn/
# End of: Referenced/Used: https://www.geeksforgeeks.org/machine-learning/confusion-matrix-machine-learning/
# End of: Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesClassifier.html
# End of: Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html
# End of: Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html
# End of: Referenced/Used: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
# End of: Referenced/Used: https://www.geeksforgeeks.org/machine-learning/f1-score-in-machine-learning/
# End of: Referenced/Used AI generated code and info from GPT-4.1 through Github Copilot (General help, Help with parse_args)
# End of: Referenced/Used material from AI healthcare class
# End of: Referenced/Used previous work completed by me for AI healthcare class

In [ ]:
# Code Block 5
# Choose between ada_boost, random_forest, and bagging
tutorial("ada_boost")